<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/anpd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"


In [2]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()


✅ Banco e tabela 'articles' prontos!


In [3]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()


In [4]:
def montar_url(pagina):
    if pagina == 1:
        return "https://www12.senado.leg.br/noticias/ultimas"
    return f"https://www12.senado.leg.br/noticias/ultimas/{pagina}"


In [5]:
noticias = []

for pagina in range(18, 0, -1):
    url = montar_url(pagina)
    print(f"📄 Coletando página {pagina}: {url}")

    r = requests.get(url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    lista = soup.find("ol", class_="lista-resultados")
    if not lista:
        print("⚠️ Nenhuma lista encontrada")
        continue

    itens = lista.find_all("li")
    print(f"   {len(itens)} notícias encontradas")

    for item in itens:
        titulo_tag = item.find("span", class_="eta")
        link_tag = item.find("a")
        data_tag = item.select_one("div.text-muted.normalis")

        if not titulo_tag or not link_tag:
            continue

        data = {
            "title": titulo_tag.get_text(strip=True),
            "date": data_tag.get_text(strip=True) if data_tag else None,
            "author": "Agência Senado",
            "url": "https://www12.senado.leg.br" + link_tag["href"],
            "source": "Senado Federal"
        }

        noticias.append(data)
        insert_article(**data)

    time.sleep(1)

print(f"\n✅ Total coletado: {len(noticias)} notícias")
df_senado = pd.DataFrame(noticias)
display(df_senado.head())


📄 Coletando página 18: https://www12.senado.leg.br/noticias/ultimas/18
   20 notícias encontradas
📄 Coletando página 17: https://www12.senado.leg.br/noticias/ultimas/17
   20 notícias encontradas
📄 Coletando página 16: https://www12.senado.leg.br/noticias/ultimas/16
   20 notícias encontradas
📄 Coletando página 15: https://www12.senado.leg.br/noticias/ultimas/15
   20 notícias encontradas
📄 Coletando página 14: https://www12.senado.leg.br/noticias/ultimas/14
   20 notícias encontradas
📄 Coletando página 13: https://www12.senado.leg.br/noticias/ultimas/13
   20 notícias encontradas
📄 Coletando página 12: https://www12.senado.leg.br/noticias/ultimas/12
   20 notícias encontradas
📄 Coletando página 11: https://www12.senado.leg.br/noticias/ultimas/11
   20 notícias encontradas
📄 Coletando página 10: https://www12.senado.leg.br/noticias/ultimas/10
   20 notícias encontradas
📄 Coletando página 9: https://www12.senado.leg.br/noticias/ultimas/9
   20 notícias encontradas
📄 Coletando página 8: 

,title,date,author,url,source
0,Paulo Paim: Homens também têm de participar do...,23/03/2026 18h10,Agência Senado,https://www12.senado.leg.br/noticias/audios/20...,Senado Federal
1,IFI chama atenção para aumento de pagamentos c...,23/03/2026 17h56,Agência Senado,https://www12.senado.leg.br/noticias/videos/20...,Senado Federal
2,Damares pede prorrogação de CPMI e prisão domi...,23/03/2026 17h32,Agência Senado,https://www12.senado.leg.br/noticias/materias/...,Senado Federal
3,Debate aponta necessidade de educação e políti...,23/03/2026 17h21,Agência Senado,https://www12.senado.leg.br/noticias/materias/...,Senado Federal
4,Izalci destaca iniciativas de apoio a ciência ...,23/03/2026 17h18,Agência Senado,https://www12.senado.leg.br/noticias/materias/...,Senado Federal


In [6]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT * FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head(20))


📦 Total no banco: 337 registros


,id,title,date,author,url,source
0,232,Plenário aprova criação de cargos na Justiça F...,31/03/2026 19h30,Agência Senado,https://www12.senado.leg.br/noticias/materias/...,Senado Federal
1,233,Já é lei a ampliação gradual da licença-patern...,31/03/2026 19h26,Agência Senado,https://www12.senado.leg.br/noticias/audios/20...,Senado Federal
2,234,Senado aprova acordo sobre ciência e tecnologi...,31/03/2026 19h21,Agência Senado,https://www12.senado.leg.br/noticias/audios/20...,Senado Federal
3,235,Diploma Bertha Lutz homenageia 15 pessoas por ...,31/03/2026 19h19,Agência Senado,https://www12.senado.leg.br/noticias/audios/20...,Senado Federal
4,236,Senado aprova reajuste para forças de seguranç...,31/03/2026 19h16,Agência Senado,https://www12.senado.leg.br/noticias/audios/20...,Senado Federal
5,237,Reajuste para forças de segurança de DF e ex-t...,31/03/2026 19h04,Agência Senado,https://www12.senado.leg.br/noticias/materias/...,Senado Federal
6,238,Senado fará homenagem a corretores de imóveis ...,31/03/2026 18h59,Agência Senado,https://www12.senado.leg.br/noticias/materias/...,Senado Federal
7,239,Senado aprova guarda compartilhada de animais ...,31/03/2026 18h48,Agência Senado,https://www12.senado.leg.br/noticias/materias/...,Senado Federal
8,240,Zenaide Maia destaca importância do novo Plano...,31/03/2026 18h26,Agência Senado,https://www12.senado.leg.br/noticias/materias/...,Senado Federal
9,241,18 de junho passa a ser o Dia Nacional do Orgu...,31/03/2026 18h23,Agência Senado,https://www12.senado.leg.br/noticias/audios/20...,Senado Federal


In [14]:
def plot_charts(df):
    if df.empty:
        print("❌ Sem dados para plotar")
        return

    # Top 15
    top15 = df.head(15).copy()
    top15["rank"] = range(1, len(top15) + 1)

    fig1 = px.bar(
        top15,
        x="rank",
        y="title",
        orientation="h",
        title="Top 15 Notícias – Internet Governance"
    )
    fig1.update_layout(height=600)
    fig1.show()
    display(fig1)

    # Fonte
    source_count = df["source"].value_counts().reset_index()
    source_count.columns = ["source", "count"]

    fig2 = px.pie(
        source_count,
        names="source",
        values="count",
        title="Distribuição por Fonte"
    )
    fig2.show()
    display(fig2)

    # Palavras
    text = " ".join(df["title"].astype(str)).lower()
    words = re.findall(r"\b\w{4,}\b", text)

    word_freq = (
        pd.Series(words)
        .value_counts()
        .head(20)
        .reset_index()
    )
    word_freq.columns = ["palavra", "freq"]

    fig3 = px.treemap(
        word_freq,
        path=["palavra"],
        values="freq",
        title="Palavras mais frequentes nos títulos"
    )
    fig3.show()
    display(fig3)

In [15]:
plot_charts(df_db)

In [16]:
df_db['parsed_date'] = pd.to_datetime(df_db['date'], format='%d/%m/%Y %Hh%M')
daily_article_count = df_db['parsed_date'].dt.date.value_counts().sort_index().reset_index()
daily_article_count.columns = ['date', 'count']

fig = px.line(
    daily_article_count,
    x='date',
    y='count',
    title='Frequência de Notícias por Data',
    labels={'date': 'Data', 'count': 'Número de Notícias'}
)
fig.update_xaxes(dtick="D1", tickformat="%d/%m/%Y") # Format x-axis to show dates clearly
fig.update_layout(hovermode="x unified")
fig.show()
display(fig)